In [1]:
!pip install gymnasium numpy -q

import gymnasium as gym
import numpy as np
import random
import copy

# Custom CartPole wrapper that lets us change physics mid-run to force adaptation
class ShiftableCartPole:
    def __init__(self):
        self.env = gym.make("CartPole-v1")
        self.env.reset()
        # store originals so we can shift away from them later
        self.default_gravity = self.env.unwrapped.gravity
        self.default_length = self.env.unwrapped.length
        self.default_masspole = self.env.unwrapped.masspole

    def set_phase(self, phase):
        # Phase 1: standard physics. Phase 2: harder, shifted physics.
        if phase == 1:
            self.env.unwrapped.gravity = self.default_gravity
            self.env.unwrapped.length = self.default_length
            self.env.unwrapped.masspole = self.default_masspole
        elif phase == 2:
            self.env.unwrapped.gravity = self.default_gravity * 1.6   # stronger gravity
            self.env.unwrapped.length = self.default_length * 1.8     # longer pole (harder to balance)
            self.env.unwrapped.masspole = self.default_masspole * 1.5

    def reset(self):
        obs, _ = self.env.reset()
        return obs

    def step(self, action):
        obs, reward, terminated, truncated, _ = self.env.step(action)
        return obs, reward, terminated or truncated

env_wrapper = ShiftableCartPole()
print("Default physics -> gravity:", env_wrapper.default_gravity,
      "length:", env_wrapper.default_length,
      "masspole:", env_wrapper.default_masspole)

Default physics -> gravity: 9.8 length: 0.5 masspole: 0.1


In [2]:
class Genome:
    def __init__(self, n_inputs=4, n_outputs=2):
        self.n_inputs = n_inputs
        self.n_outputs = n_outputs
        self.neurons = {}   # id -> {'type': str, 'bias': float}
        self.connections = []  # list of dicts: from, to, weight, enabled
        self.next_id = 0

        for _ in range(n_inputs):
            self._add_neuron('input')
        for _ in range(n_outputs):
            self._add_neuron('standard')

        # start fully connected input->output, small random weights
        input_ids = list(range(n_inputs))
        output_ids = list(range(n_inputs, n_inputs + n_outputs))
        for i in input_ids:
            for o in output_ids:
                self.connections.append({'from': i, 'to': o, 'weight': np.random.uniform(-1, 1), 'enabled': True})

    def _add_neuron(self, ntype):
        nid = self.next_id
        self.neurons[nid] = {'type': ntype, 'bias': np.random.uniform(-0.5, 0.5)}
        self.next_id += 1
        return nid

    def copy(self):
        g = Genome.__new__(Genome)
        g.n_inputs = self.n_inputs
        g.n_outputs = self.n_outputs
        g.neurons = copy.deepcopy(self.neurons)
        g.connections = copy.deepcopy(self.connections)
        g.next_id = self.next_id
        return g


class Network:
    """Executes a Genome. Neurons ordered by id = topological order
    (new neurons always get higher ids and only connect from lower ids,
    which keeps the graph a DAG and the forward pass a single sorted sweep).
    Memory neurons additionally blend in their own previous output (recurrence)."""

    def __init__(self, genome):
        self.genome = genome
        self.state = {nid: 0.0 for nid in genome.neurons}  # memory neuron persistence

    def reset_state(self):
        self.state = {nid: 0.0 for nid in self.genome.neurons}

    def activate(self, inputs):
        values = {}
        for i, nid in enumerate(range(self.genome.n_inputs)):
            values[nid] = inputs[i]

        ordered = sorted(self.genome.neurons.keys())
        for nid in ordered:
            if nid < self.genome.n_inputs:
                continue
            ntype = self.genome.neurons[nid]['type']
            bias = self.genome.neurons[nid]['bias']
            total = bias
            for c in self.genome.connections:
                if c['to'] == nid and c['enabled'] and c['from'] in values:
                    total += values[c['from']] * c['weight']

            if ntype == 'memory':
                retention = 0.6
                out = np.tanh(total) * (1 - retention) + self.state[nid] * retention
                self.state[nid] = out
            elif ntype == 'control':
                gate = 1 / (1 + np.exp(-total))
                out = total if gate > 0.5 else 0.0
            else:  # standard
                out = np.tanh(total)

            values[nid] = out

        output_ids = list(range(self.genome.n_inputs, self.genome.n_inputs + self.genome.n_outputs))
        return [values.get(o, 0.0) for o in output_ids]


# quick sanity check
g = Genome()
net = Network(g)
out = net.activate([0.1, -0.2, 0.05, 0.3])
print("Sample genome neurons:", g.neurons)
print("Sample output:", out)

Sample genome neurons: {0: {'type': 'input', 'bias': 0.20336463162403695}, 1: {'type': 'input', 'bias': 0.37505245802808596}, 2: {'type': 'input', 'bias': -0.3802219923809502}, 3: {'type': 'input', 'bias': 0.11069129056754246}, 4: {'type': 'standard', 'bias': -0.3479019099171935}, 5: {'type': 'standard', 'bias': 0.19495786681678873}}
Sample output: [np.float64(-0.49223945358046206), np.float64(-0.09423564413695465)]


In [3]:
def mutate(genome, rate=0.3):
    g = genome.copy()

    # 1. Mutate weights
    for c in g.connections:
        if random.random() < 0.8:
            c['weight'] += np.random.normal(0, 0.4)

    # 2. Add a connection
    if random.random() < rate:
        candidates = list(g.neurons.keys())
        frm = random.choice(candidates)
        to_candidates = [n for n in candidates if n >= g.n_inputs and n != frm]
        if to_candidates:
            to = random.choice(to_candidates)
            g.connections.append({'from': frm, 'to': to, 'weight': np.random.uniform(-1, 1), 'enabled': True})

    # 3. Add a neuron (with a chance of being memory or control - unified neuron model)
    if random.random() < rate * 0.5:
        ntype = random.choices(['standard', 'memory', 'control'], weights=[0.5, 0.25, 0.25])[0]
        new_id = g._add_neuron(ntype)
        existing = [n for n in g.neurons if n != new_id]
        frm = random.choice(existing)
        to = random.choice([n for n in existing if n >= g.n_inputs] + [new_id])
        g.connections.append({'from': frm, 'to': new_id, 'weight': np.random.uniform(-1, 1), 'enabled': True})
        if to != new_id:
            g.connections.append({'from': new_id, 'to': to, 'weight': np.random.uniform(-1, 1), 'enabled': True})

    # 4. Toggle a connection off/on
    if g.connections and random.random() < rate * 0.3:
        c = random.choice(g.connections)
        c['enabled'] = not c['enabled']

    return g


def compute_spectrum(genome):
    """Chromosome spectrum: a feature vector describing HOW the genome solves
    the problem structurally, independent of fitness. Used to preserve diversity
    of *approaches*, not just diversity of performance."""
    n_standard = sum(1 for n in genome.neurons.values() if n['type'] == 'standard')
    n_memory = sum(1 for n in genome.neurons.values() if n['type'] == 'memory')
    n_control = sum(1 for n in genome.neurons.values() if n['type'] == 'control')
    n_conn = sum(1 for c in genome.connections if c['enabled'])
    avg_weight = np.mean([abs(c['weight']) for c in genome.connections]) if genome.connections else 0
    n_neurons = len(genome.neurons)
    return np.array([n_standard, n_memory, n_control, n_conn, avg_weight, n_neurons])


def spectrum_distance(spec_a, spec_b):
    return np.linalg.norm(spec_a - spec_b)


# sanity check
g2 = mutate(g, rate=1.0)
print("Mutated neurons:", g2.neurons)
print("Mutated connections count:", len(g2.connections))
print("Spectrum:", compute_spectrum(g2))

Mutated neurons: {0: {'type': 'input', 'bias': 0.20336463162403695}, 1: {'type': 'input', 'bias': 0.37505245802808596}, 2: {'type': 'input', 'bias': -0.3802219923809502}, 3: {'type': 'input', 'bias': 0.11069129056754246}, 4: {'type': 'standard', 'bias': -0.3479019099171935}, 5: {'type': 'standard', 'bias': 0.19495786681678873}, 6: {'type': 'memory', 'bias': 0.2749288138855054}}
Mutated connections count: 11
Spectrum: [ 2.          1.          0.         10.          0.48161417  7.        ]


In [4]:
def evaluate_genome(genome, env_wrapper, episodes=3, max_steps=500):
    net = Network(genome)
    total_reward = 0
    for _ in range(episodes):
        net.reset_state()
        obs = env_wrapper.reset()
        ep_reward = 0
        for _ in range(max_steps):
            out = net.activate(obs)
            action = 0 if out[0] > out[1] else 1
            obs, reward, done = env_wrapper.step(action)
            ep_reward += reward
            if done:
                break
        total_reward += ep_reward
    return total_reward / episodes


def select_next_generation(population, fitnesses, pop_size):
    """Selection blends fitness with spectrum diversity: top half by fitness
    survive outright, the other half is filled by picking genomes whose
    spectrum is most different from what's already selected — this is the
    spectrum-diversity mechanism, preventing the population from collapsing
    onto one structural 'idea' even under strong selection pressure."""
    ranked = sorted(zip(population, fitnesses), key=lambda x: -x[1])
    n_elite = pop_size // 2
    survivors = [g for g, f in ranked[:n_elite]]

    remaining = [g for g, f in ranked[n_elite:]]
    survivor_specs = [compute_spectrum(g) for g in survivors]

    while len(survivors) < pop_size and remaining:
        best_idx, best_score = 0, -1
        for i, cand in enumerate(remaining):
            cand_spec = compute_spectrum(cand)
            min_dist = min(spectrum_distance(cand_spec, s) for s in survivor_specs) if survivor_specs else 0
            if min_dist > best_score:
                best_score, best_idx = min_dist, i
        chosen = remaining.pop(best_idx)
        survivors.append(chosen)
        survivor_specs.append(compute_spectrum(chosen))

    return survivors


def run_evolution(env_wrapper, generations, pop_size=30, mutation_rate=0.3, log_every=5, start_pop=None):
    population = start_pop if start_pop else [Genome() for _ in range(pop_size)]
    history = []

    for gen in range(generations):
        fitnesses = [evaluate_genome(g, env_wrapper) for g in population]
        best_fit = max(fitnesses)
        avg_fit = np.mean(fitnesses)
        history.append({'gen': gen, 'best': best_fit, 'avg': avg_fit})

        if gen % log_every == 0:
            print(f"Gen {gen}: best={best_fit:.1f}, avg={avg_fit:.1f}")

        survivors = select_next_generation(population, fitnesses, pop_size // 2)
        next_pop = survivors[:]
        while len(next_pop) < pop_size:
            parent = random.choice(survivors)
            next_pop.append(mutate(parent, mutation_rate))
        population = next_pop

    return population, history


# quick smoke test: 5 generations, small population, phase 1 (standard physics)
env_wrapper.set_phase(1)
test_pop, test_history = run_evolution(env_wrapper, generations=5, pop_size=15, log_every=1)

Gen 0: best=99.3, avg=23.8
Gen 1: best=88.3, avg=26.8
Gen 2: best=87.7, avg=27.1
Gen 3: best=167.0, avg=50.4
Gen 4: best=203.3, avg=49.6


In [5]:
random.seed(42)
np.random.seed(42)

POP_SIZE = 40
PHASE1_GENS = 40
PHASE2_GENS = 40

print("=== PHASE 1: Standard CartPole physics ===")
env_wrapper.set_phase(1)
population, phase1_history = run_evolution(env_wrapper, generations=PHASE1_GENS, pop_size=POP_SIZE, log_every=5)

print("\n=== PHASE 2: Shifted physics (gravity x1.6, pole length x1.8, mass x1.5) ===")
print("Population NOT reset - continuing evolution on the SAME genomes from Phase 1")
env_wrapper.set_phase(2)
population, phase2_history = run_evolution(env_wrapper, generations=PHASE2_GENS, pop_size=POP_SIZE,
                                             log_every=5, start_pop=population)

print(f"\nPhase 1 final best: {phase1_history[-1]['best']:.1f}")
print(f"Phase 2 gen 0 best (immediately after shift): {phase2_history[0]['best']:.1f}")
print(f"Phase 2 final best: {phase2_history[-1]['best']:.1f}")

=== PHASE 1: Standard CartPole physics ===
Gen 0: best=161.0, avg=22.9
Gen 5: best=500.0, avg=167.3
Gen 10: best=500.0, avg=229.4
Gen 15: best=500.0, avg=243.0
Gen 20: best=500.0, avg=259.0
Gen 25: best=500.0, avg=265.9
Gen 30: best=500.0, avg=240.3
Gen 35: best=500.0, avg=240.7

=== PHASE 2: Shifted physics (gravity x1.6, pole length x1.8, mass x1.5) ===
Population NOT reset - continuing evolution on the SAME genomes from Phase 1
Gen 0: best=500.0, avg=225.1
Gen 5: best=500.0, avg=222.6
Gen 10: best=500.0, avg=266.0
Gen 15: best=500.0, avg=221.6
Gen 20: best=500.0, avg=202.2
Gen 25: best=500.0, avg=234.2
Gen 30: best=500.0, avg=237.2
Gen 35: best=500.0, avg=213.1

Phase 1 final best: 500.0
Phase 2 gen 0 best (immediately after shift): 500.0
Phase 2 final best: 500.0


In [6]:
# Get the single best genome from END of Phase 1
phase1_fitnesses = [evaluate_genome(g, env_wrapper, episodes=10) for g in population]
# NOTE: population here has already evolved through Phase 2 - we need to re-run
# to isolate the Phase 1 champion specifically. Let's redo this properly:

random.seed(42)
np.random.seed(42)

env_wrapper.set_phase(1)
pop_p1, hist_p1 = run_evolution(env_wrapper, generations=PHASE1_GENS, pop_size=POP_SIZE, log_every=10)

# Identify Phase 1 champion with a robust (10-episode) evaluation
p1_fitnesses = [evaluate_genome(g, env_wrapper, episodes=10) for g in pop_p1]
champion_p1 = pop_p1[np.argmax(p1_fitnesses)]
print(f"\nPhase 1 champion fitness (on Phase 1 physics, 10-ep avg): {max(p1_fitnesses):.1f}")

# Test the SAME champion, unmodified, directly on Phase 2 physics (no adaptation yet)
env_wrapper.set_phase(2)
champion_on_shifted = evaluate_genome(champion_p1, env_wrapper, episodes=10)
print(f"Same champion tested on Phase 2 physics (no adaptation): {champion_on_shifted:.1f}")

# Baseline: a freshly random-initialized genome on Phase 2 physics
random_genome = Genome()
random_baseline = evaluate_genome(random_genome, env_wrapper, episodes=10)
print(f"Random fresh genome on Phase 2 physics (baseline floor): {random_baseline:.1f}")

# Now evolve starting FROM the Phase 1 population, on Phase 2 physics
pop_p2, hist_p2 = run_evolution(env_wrapper, generations=PHASE2_GENS, pop_size=POP_SIZE,
                                  log_every=10, start_pop=pop_p1)
p2_fitnesses = [evaluate_genome(g, env_wrapper, episodes=10) for g in pop_p2]
print(f"\nAfter {PHASE2_GENS} generations adapting to Phase 2: best={max(p2_fitnesses):.1f}, avg={np.mean(p2_fitnesses):.1f}")

Gen 0: best=131.3, avg=24.3
Gen 10: best=500.0, avg=195.5
Gen 20: best=500.0, avg=224.0
Gen 30: best=500.0, avg=222.2

Phase 1 champion fitness (on Phase 1 physics, 10-ep avg): 500.0
Same champion tested on Phase 2 physics (no adaptation): 34.3
Random fresh genome on Phase 2 physics (baseline floor): 29.7
Gen 0: best=500.0, avg=148.8
Gen 10: best=500.0, avg=242.8
Gen 20: best=500.0, avg=252.5
Gen 30: best=500.0, avg=235.2

After 40 generations adapting to Phase 2: best=500.0, avg=224.1


In [7]:
random.seed(123)
np.random.seed(123)

print("=== COLD-START BASELINE: fresh population trained ONLY on Phase 2 physics ===")
env_wrapper.set_phase(2)
pop_cold, hist_cold = run_evolution(env_wrapper, generations=PHASE2_GENS, pop_size=POP_SIZE, log_every=5)

# Compare: generations needed to first reach avg fitness >= 150
def gens_to_threshold(history, threshold=150):
    for h in history:
        if h['avg'] >= threshold:
            return h['gen']
    return None

warm_start_gen = gens_to_threshold(hist_p2)
cold_start_gen = gens_to_threshold(hist_cold)

print(f"\nWarm-start (from Phase 1 population) reached avg>=150 at gen: {warm_start_gen}")
print(f"Cold-start (fresh random population) reached avg>=150 at gen: {cold_start_gen}")
print(f"\nWarm-start final avg: {hist_p2[-1]['avg']:.1f}")
print(f"Cold-start final avg: {hist_cold[-1]['avg']:.1f}")

=== COLD-START BASELINE: fresh population trained ONLY on Phase 2 physics ===
Gen 0: best=81.3, avg=20.1
Gen 5: best=358.3, avg=65.9
Gen 10: best=484.0, avg=99.7
Gen 15: best=500.0, avg=173.9
Gen 20: best=500.0, avg=244.6
Gen 25: best=500.0, avg=216.4
Gen 30: best=500.0, avg=207.5
Gen 35: best=500.0, avg=220.8

Warm-start (from Phase 1 population) reached avg>=150 at gen: 1
Cold-start (fresh random population) reached avg>=150 at gen: 11

Warm-start final avg: 228.9
Cold-start final avg: 192.5


In [8]:
def run_evolution_adaptive(env_wrapper, generations, pop_size=30, base_rate=0.3,
                             boost_rate=0.7, boost_decay=0.85, log_every=5, start_pop=None):
    population = start_pop if start_pop else [Genome() for _ in range(pop_size)]
    history = []
    current_rate = base_rate
    prev_avg = None

    for gen in range(generations):
        fitnesses = [evaluate_genome(g, env_wrapper) for g in population]
        best_fit = max(fitnesses)
        avg_fit = np.mean(fitnesses)

        # Shift detection: if avg fitness drops sharply vs last gen, spike mutation rate
        if prev_avg is not None and avg_fit < prev_avg * 0.5:
            current_rate = boost_rate
        else:
            current_rate = max(base_rate, current_rate * boost_decay)
        prev_avg = avg_fit

        history.append({'gen': gen, 'best': best_fit, 'avg': avg_fit, 'mut_rate': current_rate})
        if gen % log_every == 0:
            print(f"Gen {gen}: best={best_fit:.1f}, avg={avg_fit:.1f}, mut_rate={current_rate:.2f}")

        survivors = select_next_generation(population, fitnesses, pop_size // 2)
        next_pop = survivors[:]
        while len(next_pop) < pop_size:
            parent = random.choice(survivors)
            next_pop.append(mutate(parent, current_rate))
        population = next_pop

    return population, history


random.seed(42)
np.random.seed(42)

print("=== IMPROVED: warm-start with adaptive mutation rate on Phase 2 ===")
env_wrapper.set_phase(2)
pop_adaptive, hist_adaptive = run_evolution_adaptive(env_wrapper, generations=PHASE2_GENS,
                                                        pop_size=POP_SIZE, log_every=5, start_pop=pop_p1)

warm_adaptive_gen = gens_to_threshold(hist_adaptive)
print(f"\nWarm-start + adaptive mutation reached avg>=150 at gen: {warm_adaptive_gen}")
print(f"Warm-start + adaptive mutation final avg: {hist_adaptive[-1]['avg']:.1f}")
print(f"\nComparison summary:")
print(f"  Cold-start:              gen {cold_start_gen} to threshold, final avg {hist_cold[-1]['avg']:.1f}")
print(f"  Warm-start (fixed rate): gen {warm_start_gen} to threshold, final avg {hist_p2[-1]['avg']:.1f}")
print(f"  Warm-start (adaptive):   gen {warm_adaptive_gen} to threshold, final avg {hist_adaptive[-1]['avg']:.1f}")

=== IMPROVED: warm-start with adaptive mutation rate on Phase 2 ===
Gen 0: best=500.0, avg=146.8, mut_rate=0.30
Gen 5: best=500.0, avg=174.8, mut_rate=0.30
Gen 10: best=500.0, avg=266.1, mut_rate=0.30
Gen 15: best=500.0, avg=235.6, mut_rate=0.30
Gen 20: best=500.0, avg=229.4, mut_rate=0.30
Gen 25: best=500.0, avg=194.6, mut_rate=0.30
Gen 30: best=500.0, avg=211.8, mut_rate=0.30
Gen 35: best=500.0, avg=185.2, mut_rate=0.30

Warm-start + adaptive mutation reached avg>=150 at gen: 1
Warm-start + adaptive mutation final avg: 205.0

Comparison summary:
  Cold-start:              gen 11 to threshold, final avg 192.5
  Warm-start (fixed rate): gen 1 to threshold, final avg 228.9
  Warm-start (adaptive):   gen 1 to threshold, final avg 205.0
